# Week 25, Day 2: Principal components of a population

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w025/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.load(dataset("churchland2012_reach.npz"))
rates, times = data["rates"].astype(float), data["times"]
C, T, N = rates.shape
Xn = rates / (rates.max(axis=(0, 1)) - rates.min(axis=(0, 1)) + 5)     # soft normalisation
X = Xn - Xn.mean(axis=0, keepdims=True)                                 # remove the cross-condition mean
win = (times >= -50) & (times <= 150)
Xw = X[:, win, :]                                                       # conditions x 21 times x neurons
D = Xw.reshape(-1, N)                                                   # 2268 population states x 218 neurons
print("data matrix:", D.shape)

### Exercise: PCA two ways

Centre D by subtracting the mean of each neuron (column), take its SVD, and store the principal directions in `Vt` and the fraction of variance explained by each component in `ve`. The test compares your answer with the eigenvalues of the covariance matrix.

In [ ]:
mu = D.mean(axis=0)
U, S, Vt = ...
ve = ...
print("variance explained by the first six components:", ve[:6].round(3), " total", ve[:6].sum().round(3))

In [ ]:
import numpy as np
def test_ve():
    "ve holds the fraction of variance per component, decreasing and summing to 1"
    assert abs(ve.sum() - 1) < 1e-9 and np.all(np.diff(ve) <= 1e-12)

def test_eigh():
    "the SVD variances match the eigenvalues of the covariance matrix"
    lam = np.sort(np.linalg.eigvalsh(np.cov(D, rowvar=False)))[::-1]
    assert np.allclose(ve[:10], lam[:10] / lam.sum(), atol=1e-9)

def test_V():
    "the rows of Vt are orthonormal"
    assert np.allclose(Vt[:6] @ Vt[:6].T, np.eye(6), atol=1e-9)

test_ve()
test_eigh()
test_V()
print("All checks passed")

### Exercise: Two measures of dimensionality

Compute `k80`, the number of components needed to explain 80% of the variance, and `pr`, the participation ratio of the eigenvalues (S²).

In [ ]:
k80 = ...
lam = S ** 2
pr = ...
print(f"{k80} components reach 80% of the variance;  participation ratio {pr:.1f};  out of {N} neurons")

In [ ]:
import numpy as np
def test_k80():
    "k80 is the smallest number of components whose cumulative variance reaches 80%"
    cum = np.cumsum(ve)
    assert cum[k80 - 1] >= 0.8 and (k80 == 1 or cum[k80 - 2] < 0.8)

def test_pr():
    "pr is the participation ratio of the eigenvalues"
    lam = S ** 2
    assert abs(pr - lam.sum() ** 2 / (lam ** 2).sum()) < 1e-9

test_k80()
test_pr()
print("All checks passed")

### Exercise: Population trajectories

Project the windowed data onto the first six principal directions to get `Z` (conditions × 21 × 6), then plot every condition's trajectory in PC1 against PC2, marking its starting point.

In [ ]:
Z = ...
fig, ax = plt.subplots(figsize=(5, 5))
for c in range(C):
    ax.plot(Z[c, :, 0], Z[c, :, 1], lw=0.7)
    ax.plot(Z[c, 0, 0], Z[c, 0, 1], "k.", ms=3)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_aspect("equal"); plt.show()
print("projected variances:", Z.reshape(-1, 6).var(axis=0).round(4))

In [ ]:
import numpy as np
def test_Z():
    "Z holds the projections onto the first six components, shape conditions x 21 x 6"
    assert Z.shape == (C, 21, 6) and np.allclose(Z, (Xw - mu) @ Vt[:6].T)

def test_var():
    "the variance of each projected component equals its eigenvalue divided by the number of states"
    assert np.allclose(Z.reshape(-1, 6).var(axis=0), S[:6] ** 2 / D.shape[0])

test_Z()
test_var()
print("All checks passed")